# Filtering alignments by window sp_score

Run this notebook using the cog2026.10.22_En0.7.9 conda environment.

Defines `filter_windows_spscore`, a `scinexus` composable app that keeps only the
windows of an alignment whose sum-of-pairs score (cogent3 `sp_score`) is above a
threshold, and returns the kept windows joined end to end. It reproduces the
clean-blocks subset built in `alignment_quality.ipynb` in a single app, so it can be
chained after `load_aligned` and the renamers in `libs.py`.

The threshold is the `sp_score` of a synthetic alignment of `window` columns in which
every sequence is all `A` except one, which has `round(window * threshold_filter)`
bases followed by gaps. It is computed for the number of sequences and the names of
the alignment being filtered (with three sequences and `window=60`,
`threshold_filter=0.5` it is 36; with four sequences it is 144). Windows that
`sp_score` cannot score are dropped. If no window passes, the app returns
`NotCompleted`, as cogent3's own `omit_gap_pos` does for an empty result.


In [1]:
from pathlib import Path

import numpy as np
from cogent3 import get_app, load_aligned_seqs, make_aligned_seqs
from cogent3.app.typing import AlignedSeqsType
from scinexus.composable import NotCompleted, define_app

import paths
import libs


## Checks

1. On a hand-made alignment, the window with a species fully gapped is dropped and the
   fully-gapped alignment returns `NotCompleted`.
2. On the chr22 `intergenicAR` fragments the app reproduces the clean-blocks subset of
   `alignment_quality.ipynb` (`WINDOW = 60`, `THRESHOLD_FILTER = 0.5`): same columns per
   file as `windows_subset` / `subset_alns` there, and the same total kept columns.


In [3]:
app = libs.filter_windows_spscore(window=10, threshold_filter=0.5)

good, bad = "ACGTACGTAC", "----------"
toy = make_aligned_seqs(
    {"Human": good * 3, "Chimpanzee": good * 3, "Orangutan": good + bad + good},
    moltype="dna",
)
out = app(toy)
print("toy: 30 columns ->", len(out), "kept (middle window dropped):", out.to_dict() == {n: good * 2 for n in toy.names})

all_gapped = make_aligned_seqs({"Human": good, "Chimpanzee": bad, "Orangutan": bad}, moltype="dna")
print("all gapped ->", app(all_gapped))
out


toy: 30 columns -> 20 kept (middle window dropped): True
all gapped -> NotCompleted(type=FAIL, origin=filter_windows_spscore, source="unknown", message="no window above the sp_score threshold")


,0
Chimpanzee,ACGTACGTACACGTACGTAC
Human,....................
Orangutan,....................


In [ ]:
WINDOW, THRESHOLD_FILTER = 60, 0.5
data_dir = Path(paths.DATA_HUMCHIMPORANGOR114) / "intergenicAR/nonmasked_alldata_chrm22"
app = libs.filter_windows_spscore(window=WINDOW, threshold_filter=THRESHOLD_FILTER)

fnames = sorted(p for p in data_dir.glob("*.fa") if p.is_file())
kept_cols, n_completed, n_not_completed = 0, 0, 0
for fname in fnames:
    result = app(load_aligned_seqs(fname, moltype="dna"))
    if result:
        kept_cols += len(result)
        n_completed += 1
    else:
        n_not_completed += 1
print(f"files: {len(fnames)}  with kept windows: {n_completed}  NotCompleted: {n_not_completed}")
print(f"columns kept: {kept_cols:,}  (alignment_quality.ipynb reports the same number in cell 9)")


files: 551  with kept windows: 504  NotCompleted: 47
columns kept: 2,350,680  (alignment_quality.ipynb reports the same number in cell 9)
